# Coleta de notícias sobre Juizados Especiais

Usei Beautiful Soup para retirar as dez notícias mais recentes da página dos Juizados Especiais no portal do CNJ. As notícias complementam o indicador estatístico com informações de contexto.

In [1]:
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

#feito com auxilio de IA chat GPT

PASTA_PROJETO = Path.cwd()
if PASTA_PROJETO.name == "src":
    PASTA_PROJETO = PASTA_PROJETO.parent
ARQUIVO_BRUTO = PASTA_PROJETO / "data" / "raw" / "noticias_juizados.html"
ARQUIVO_TRATADO = PASTA_PROJETO / "data" / "processed" / "noticias_juizados.csv"

In [2]:
resposta = requests.get(
    "https://www.cnj.jus.br/programas-e-acoes/juizados-especiais/noticias/",
    headers={"User-Agent": "Mozilla/5.0 (projeto acadêmico TP2)"},
    timeout=120,
)
print("Status da página:", resposta.status_code)
resposta.raise_for_status()
ARQUIVO_BRUTO.write_text(resposta.text, encoding="utf-8")

Status da página: 200


284153

In [3]:
artigos = BeautifulSoup(resposta.text, "html.parser").select("article.elementor-post")[:10]
print("Notícias encontradas:", len(artigos))
if not artigos:
    raise RuntimeError("Nenhuma notícia encontrada. Verifique o seletor da página do CNJ.")
print(artigos[0].select_one(".elementor-post__title a").get_text(" ", strip=True))

Notícias encontradas: 10
Boas práticas e parceria institucional fortalecem Juizados Especiais


In [4]:
linhas = []

for artigo in artigos:
    titulo = artigo.select_one(".elementor-post__title a")
    linhas.append({
        "titulo": titulo.get_text(" ", strip=True),
        "data": artigo.select_one(".elementor-post-date").get_text(" ", strip=True),
        "resumo": artigo.select_one(".elementor-post__excerpt").get_text(" ", strip=True),
        "url": titulo["href"],
        "fonte": "Conselho Nacional de Justiça",
    })

noticias = pd.DataFrame(linhas)
noticias

,titulo,data,resumo,url,fonte
0,Boas práticas e parceria institucional fortale...,21 de setembro de 2026,O fortalecimento dos Juizados Especiais passa ...,https://www.cnj.jus.br/boas-praticas-e-parceri...,Conselho Nacional de Justiça
1,Fórum dos Juizados Especiais será realizado em...,1 de setembro de 2026,"A Paraíba vai receber, entre os dias 17 e 19 d...",https://www.cnj.jus.br/forum-dos-juizados-espe...,Conselho Nacional de Justiça
2,Juizado na Comunidade leva serviços de cidadan...,30 de junho de 2026,A Central dos Juizados Especiais do Tribunal d...,https://www.cnj.jus.br/juizado-na-comunidade-l...,Conselho Nacional de Justiça
3,Atuação dos Juizados Especiais aproxima a Just...,16 de junho de 2026,Com o tema “Fortalecer os Juizados Especiais é...,https://www.cnj.jus.br/atuacao-dos-juizados-es...,Conselho Nacional de Justiça
4,Tribunais promovem ações e mutirões processuai...,15 de junho de 2026,Com o tema “Fortalecer os Juizados Especiais é...,https://www.cnj.jus.br/tribunais-promovem-acoe...,Conselho Nacional de Justiça
5,CNJ reforça atuação em rede e inovação durante...,29 de maio de 2026,O Conselho Nacional de Justiça (CNJ) participo...,https://www.cnj.jus.br/cnj-reforca-atuacao-em-...,Conselho Nacional de Justiça
6,Juizado Especial do Torcedor e Grupo de Trabal...,7 de maio de 2026,O Juizado Especial de Defesa do Torcedor de Sã...,https://www.cnj.jus.br/juizado-especial-do-tor...,Conselho Nacional de Justiça
7,Unidade Móvel de Trânsito do TJRN fecha 2025 c...,27 de janeiro de 2026,A Unidade Móvel de Trânsito (UMT) do Tribunal ...,https://www.cnj.jus.br/unidade-movel-de-transi...,Conselho Nacional de Justiça
8,TJPB adota política para equilibrar distribuiç...,20 de janeiro de 2026,"Na última semana, o Órgão Especial do Tribunal...",https://www.cnj.jus.br/tjpb-adota-politica-par...,Conselho Nacional de Justiça
9,InfoVOO reúne dados de voos para subsidiar dem...,16 de janeiro de 2026,Uma nova solução tecnológica subsidiará princi...,https://www.cnj.jus.br/infovoo-reune-dados-de-...,Conselho Nacional de Justiça


In [5]:
assert len(noticias) == 10
assert noticias["titulo"].notna().all()
assert noticias["url"].nunique() == 10

print("Testes concluídos sem erros.")

Testes concluídos sem erros.


In [6]:
noticias.to_csv(ARQUIVO_TRATADO, index=False, encoding="utf-8-sig")
print("Arquivo bruto:", ARQUIVO_BRUTO)
print("Arquivo tratado:", ARQUIVO_TRATADO)

Arquivo bruto: c:\Projetos\Projeto de Bloco\CarolinaGabrielaMesquita_PB_TP2\TP2_Juizados_Especiais\data\raw\noticias_juizados.html
Arquivo tratado: c:\Projetos\Projeto de Bloco\CarolinaGabrielaMesquita_PB_TP2\TP2_Juizados_Especiais\data\processed\noticias_juizados.csv
